# 🌈 Rainbow DQN 공부 노트 (Seaquest 목표)

이 노트북은 **개념/메커니즘 공부용**입니다. 구현 코드는 없습니다 — *"먼저 대충 그림을 잡고, 그다음 직접 구현"* 흐름의 앞단계예요.
실제 구현 로드맵(파일별 변경점·순서·검증)은 plan 파일 `rainbow-dqn-resilient-barto.md`에 있습니다. 이 노트북은 그 plan의 **"각 요소가 도대체 무슨 메커니즘인가"**를 채워주는 짝꿍 문서입니다.

> **한 줄 정의**: Rainbow = 기본 DQN에, 서로 독립적으로 발전해 온 **6가지 개선**을 한꺼번에 합친 것. 각 개선은 DQN의 *서로 다른 약점* 하나씩을 고칩니다.

**Rainbow 논문**: Hessel et al., *"Rainbow: Combining Improvements in Deep RL"* (AAAI 2018).


## 이 노트북 사용법 & 공부 흐름

1. **0번(복습)**으로 우리가 이미 가진 DQN의 뼈대를 떠올린다 — 모든 개선은 이 뼈대의 *어디를* 건드리는지로 이해하면 쉽다.
2. **1~6번**을 순서대로 읽는다. 각 절은 항상 같은 틀:
   - 🩹 **고치는 약점** (왜 필요한가)
   - ⚙️ **메커니즘** (무엇을, 어떻게)
   - 🧮 **수식** (핵심만)
   - 🔌 **우리 코드 어디가 바뀌나** (개념 수준, 코드 아님)
   - ⚠️ **함정 / 헷갈리는 점**
3. **7번**에서 6개가 서로 어떻게 맞물리는지(특히 C51이 다른 것들과 만나는 지점)를 본다.
4. **8~9번**에서 Seaquest 적용 포인트와 공부→구현 순서/검증을 본다.

수식 기호: 상태 $s$, 행동 $a$, 보상 $r$, 할인율 $\gamma$, 온라인망 $Q_\theta$, 타깃망 $Q_{\theta^-}$.


## 0. 복습 — 우리 DQN이 이미 가진 뼈대

Rainbow를 얹기 전, 현재 코드(= Nature DQN 2015)가 무엇을 하는지 한 줄씩:

| 조각 | 현재 동작 | Rainbow에서 운명 |
|---|---|---|
| 전처리 | 흑백 84×84 + frame_skip 4 + frame stack 4 | 그대로 유지 |
| Q-network (CNN) | 화면 → 각 행동의 **Q값 1개씩** | Dueling/Noisy/C51이 **구조를 바꿈** |
| Replay Buffer | **균일 무작위** 샘플 | PER + n-step이 **샘플 방식을 바꿈** |
| 행동 선택 | **ε-greedy** | NoisyNet이 **대체(제거)** |
| 학습 타깃 | $r + \gamma \max_{a'} Q_{\theta^-}(s',a')$ | Double·n-step·C51이 **타깃 식을 바꿈** |
| target network | 주기적 복사 | 그대로 유지 |
| reward clipping | $\mathrm{sign}(r)\in\{-1,0,+1\}$ | 그대로 유지 |

**핵심 직관**: DQN의 학습은 결국 "예측 $Q_\theta(s,a)$ 를 타깃 $y$ 에 가깝게" 회귀하는 것.
$$ \text{loss} = \big(Q_\theta(s,a) - y\big)^2,\qquad y = r + \gamma \max_{a'} Q_{\theta^-}(s',a'). $$
Rainbow의 6개는 이 식의 **(1) 타깃 $y$ 계산법**, **(2) 네트워크 구조**, **(3) 어떤 샘플로 학습하나**, **(4) 어떻게 탐험하나** 를 각각 손본다.


## Rainbow 한눈에 — 6개 개선과 각자 고치는 약점

| # | 개선 | 고치는 DQN의 약점 | 한 줄 메커니즘 | 건드리는 축 |
|---|---|---|---|---|
| 1 | **Double DQN** | $\max$ 때문에 Q값을 **과대추정** | 행동 *선택*은 온라인망, *평가*는 타깃망 | 타깃 식 |
| 2 | **Dueling** | 상태가치/행동이점을 **뭉뚱그려** 비효율 | $Q = V(s) + (A(s,a)-\overline{A})$ | 네트워크 구조 |
| 3 | **n-step** | 1-step은 보상 전파가 **느림** | 보상 $n$칸 합산 후 부트스트랩 | 타깃 식 |
| 4 | **PER** | 모든 경험을 **똑같이** 샘플 → 비효율 | TD오차 큰 경험을 **자주** 샘플(+보정) | 샘플 방식 |
| 5 | **NoisyNet** | ε-greedy 탐험이 **거칠고 수동** | 가중치에 **학습되는 노이즈** → 자율 탐험 | 탐험 |
| 6 | **C51 (분포)** | 기대값 하나로 **불확실성 정보 손실** | Q값 대신 **수익의 분포**를 예측 | 네트워크 구조 + 타깃 식 |

> 외우는 팁: **"타깃 2개(Double, n-step), 구조 2개(Dueling, C51), 샘플 1개(PER), 탐험 1개(Noisy)"** — 단, C51은 구조+타깃 둘 다 건드리는 끝판왕.


## 1. Double DQN

🩹 **고치는 약점 — 과대추정(overestimation).**
타깃의 $\max_{a'} Q_{\theta^-}(s',a')$ 는 "노이즈가 낀 추정값들 중 최댓값"이다. 추정에 오차가 있으면 *최댓값은 체계적으로 위로* 치우친다(운 좋게 부풀려진 값이 뽑히기 쉬움). 그 부푼 값이 타깃이 되어 다시 학습되며 **양의 편향이 누적**된다.

⚙️ **메커니즘 — 선택과 평가의 분리.**
"다음 상태에서 *어떤* 행동이 최선인가(선택)"는 **온라인망 $Q_\theta$**가 정하고, "그 행동이 *얼마나* 좋은가(평가)"는 **타깃망 $Q_{\theta^-}$**가 매긴다. 두 망이 같은 오차를 공유하지 않으므로 부풀림이 줄어든다.

🧮 **수식 (기존 → Double):**
$$ y_{\text{DQN}} = r + \gamma\, Q_{\theta^-}\!\big(s',\, \textstyle\arg\max_{a'} Q_{\theta^-}(s',a')\big) $$
$$ y_{\text{Double}} = r + \gamma\, Q_{\theta^-}\!\big(s',\, \underbrace{\textstyle\arg\max_{a'} Q_{\theta}(s',a')}_{\text{선택은 온라인망}}\big) $$
차이는 단 하나: $\arg\max$ 안의 네트워크가 $\theta^-$ → $\theta$.

🔌 **우리 코드 어디가 바뀌나 (개념):** 학습 타깃을 만드는 곳(loss 계산) **한 군데**. `max(...)` 한 줄이 "온라인망으로 argmax → 타깃망으로 그 행동 gather" 두 단계로 바뀐다. 네트워크 구조·버퍼·루프는 그대로.

⚠️ **함정:** ① 타깃 계산은 여전히 gradient 차단 영역 안. ② Rainbow에서는 이 "행동 선택"이 나중에 C51과 만나면 *기대값* 기준 argmax가 된다(7번 참고). ③ 가장 쉬운 1번 타자라 첫 실험으로 적합.


## 2. Dueling Network

🩹 **고치는 약점 — 비효율적 표현.**
많은 상태에서는 *어떤 행동을 해도 비슷*하다(예: Seaquest에서 적이 멀리 있을 때). 그런데 일반 DQN은 행동별 Q값을 **따로따로** 배워야 해서, "이 상태 자체가 얼마나 좋은가"라는 공통 정보를 행동 수만큼 중복 학습한다.

⚙️ **메커니즘 — 두 갈래로 분해.**
CNN 특징을 두 흐름으로 나눈다:
- **가치(Value) $V(s)$**: 상태 자체의 좋음 (스칼라 1개)
- **이점(Advantage) $A(s,a)$**: 각 행동이 평균보다 *얼마나 더/덜* 좋은가 (행동 수만큼)

그리고 다시 합쳐 Q값을 만든다. 합칠 때 그냥 $V+A$로 두면 $V$와 $A$를 유일하게 나눌 수 없어(식별 불가) 학습이 흔들리므로, **이점에서 평균을 빼서** 기준점을 고정한다.

🧮 **수식:**
$$ Q(s,a) = V(s) + \Big( A(s,a) - \frac{1}{|\mathcal{A}|}\sum_{a'} A(s,a') \Big) $$
(Rainbow는 $\max$ 대신 **평균** 빼기를 사용 — 더 안정적.)

🔌 **우리 코드 어디가 바뀌나 (개념):** **네트워크 구조만**. conv 뒤 FC를 두 갈래(V용·A용)로 만들고 위 식으로 결합. **출력 shape는 그대로** $(N, \text{행동수})$ 라 학습 루프/loss는 손 안 댐.

⚠️ **함정:** ① 평균 빼기를 잊으면 학습 불안정. ② V는 출력 1, A는 출력 = 행동 수. ③ 구조만 바뀌므로 체크포인트 호환이 깨진다 → 저장 포맷에 설정을 함께 남겨야 eval에서 복원 가능(plan의 Phase 0).


## 3. Multi-step (n-step) Returns

🩹 **고치는 약점 — 느린 보상 전파.**
1-step 타깃은 보상 정보를 **한 칸씩** 뒤로 흘린다. "10스텝 전 행동이 지금 보상의 원인"이라는 걸 배우려면 그 신호가 10번의 업데이트를 거쳐 천천히 전파된다. Seaquest처럼 행동→보상 지연이 있는 게임에서 특히 느리다.

⚙️ **메커니즘 — 여러 칸을 한 번에.**
부트스트랩(미래를 네트워크 추정으로 메우기)을 1칸 뒤가 아니라 **$n$칸 뒤**에서 한다. 그 사이 $n$개의 실제 보상은 직접 더한다. 실제 보상을 더 많이 쓰니 **신호가 빨리 전파**되고 초기 편향이 준다(대신 분산은 약간 늘어 trade-off).

🧮 **수식 — $n$-step 누적보상과 타깃:**
$$ R_t^{(n)} = \sum_{k=0}^{n-1} \gamma^{k}\, r_{t+k} $$
$$ y = R_t^{(n)} + \gamma^{n} \max_{a'} Q_{\theta^-}(s_{t+n}, a') $$
($n=1$이면 원래 DQN과 동일. 표준은 $n=3$.)

🔌 **우리 코드 어디가 바뀌나 (개념):** ① 버퍼에 넣기 전, 최근 $n$개 전이를 **작은 큐**에 모아 $R_t^{(n)}$ 을 계산해 한 개의 전이 $(s_t, a_t, R_t^{(n)}, s_{t+n}, \text{done})$ 로 저장. ② 타깃에서 $\gamma$ → $\gamma^{n}$.

⚠️ **함정:** ① **에피소드 경계** — 중간에 done이 나오면 거기서 잘라야 한다(다음 에피소드 보상을 더하면 안 됨). ② reward clipping은 **누적 전에** 칸마다 적용. ③ $n$이 커질수록 분산↑ — 무작정 키우지 말 것.


## 4. Prioritized Experience Replay (PER)

🩹 **고치는 약점 — 균일 샘플의 비효율.**
기본 버퍼는 모든 경험을 **똑같은 확률**로 뽑는다. 하지만 이미 잘 맞히는 평범한 경험은 배울 게 적고, **예측이 크게 틀린 경험(=놀라운 경험)**에서 배울 게 많다. 균일 샘플은 이 귀한 경험을 자주 못 본다.

⚙️ **메커니즘 — 놀라움(TD오차)에 비례해 뽑기 + 편향 보정.**
각 전이에 우선순위 $p_i = |\delta_i| + \epsilon$ 를 매긴다($\delta_i$ = TD오차, $\epsilon$ = 0 방지용 작은 값). 우선순위에 비례해 샘플:
$$ P(i) = \frac{p_i^{\alpha}}{\sum_k p_k^{\alpha}} $$
$\alpha$ 는 "얼마나 편식할지"(0이면 균일, 1이면 완전 비례). 그런데 이렇게 *편향되게* 뽑으면 기댓값이 틀어지므로, **중요도 표본 가중치(IS weight)**로 loss를 보정한다:
$$ w_i = \Big( \frac{1}{N \cdot P(i)} \Big)^{\beta} \Big/ \max_j w_j,\qquad \text{loss} \mathrel{*}= w_i $$
$\beta$ 는 0.4 → 1.0 으로 **어닐링**(학습 후반에 보정 완전 적용). 자주 뽑힌 샘플일수록 가중치를 낮춰 균형을 맞춘다.

자료구조: 비례 샘플을 $O(\log N)$ 으로 하려고 **sum-tree**(구간합 트리)를 쓴다. 잎=각 전이 우선순위, 부모=자식 합. 0~총합 사이 난수로 트리를 내려가며 샘플.

🔌 **우리 코드 어디가 바뀌나 (개념):** 버퍼를 **재작성**(sum-tree, 우선순위 저장, IS 가중치 반환). 학습 루프는 ① loss에 $w_i$ 곱, ② 매 업데이트 후 새 TD오차로 **우선순위 갱신**, ③ 새 전이는 **최대 우선순위**로 삽입.

⚠️ **함정:** ① $\beta$ 어닐링 잊으면 편향 잔존. ② **메모리** — 현재 버퍼는 stack을 통째 저장(전이당 8프레임, 1M이면 ~56GB). 장기 Seaquest면 capacity를 줄이거나 *프레임 1장 단위 저장*으로 바꾸는 걸 PER 재작성과 함께. ③ C51을 붙인 뒤엔 "TD오차" 자리에 **샘플별 cross-entropy 손실**을 우선순위로 쓴다(7번).


## 5. Noisy Nets

🩹 **고치는 약점 — 거칠고 수동적인 ε-greedy 탐험.**
ε-greedy는 "ε 확률로 *완전 무작위* 행동"이라 비효율적이고, ε 스케줄을 사람이 손으로 짜야 한다. 또 상태와 무관하게 **균일하게** 무작위라, "여긴 더 탐험이 필요하고 저긴 아니다"를 구분 못 한다.

⚙️ **메커니즘 — 가중치에 학습되는 노이즈.**
선형층의 가중치/편향을 "평균 + 노이즈"로 둔다. 노이즈의 세기 $\sigma$ 자체를 **학습**하므로, 망이 스스로 "어디서 얼마나 흔들지(=탐험할지)"를 조절한다. 학습이 진행되면 불필요한 곳의 $\sigma$ 는 자연히 줄어 탐험이 자동으로 식는다.

🧮 **수식 — NoisyLinear:**
$$ y = (\mu^{W} + \sigma^{W} \odot \varepsilon^{W})\,x \;+\; (\mu^{b} + \sigma^{b} \odot \varepsilon^{b}) $$
여기서 $\mu, \sigma$ 는 **학습 파라미터**, $\varepsilon$ 은 매 forward마다 새로 뽑는 노이즈. 파라미터 수를 줄이려고 **factorized Gaussian**을 쓴다:
$$ \varepsilon^{W}_{ij} = f(\varepsilon_i)\,f(\varepsilon_j),\qquad f(x) = \mathrm{sgn}(x)\sqrt{|x|} $$
($\varepsilon_i, \varepsilon_j$ 는 입력/출력 차원 길이의 1D 노이즈 벡터.)

🔌 **우리 코드 어디가 바뀌나 (개념):** ① FC들(및 Dueling 두 갈래)을 `NoisyLinear`로 교체. ② **ε-greedy 완전 제거** — 행동 선택은 그냥 $\arg\max Q$. 즉 행동선택 함수에서 epsilon 인자가 사라진다 → 이를 쓰는 play/eval 코드도 따라 수정.

⚠️ **함정:** ① 학습 중엔 매 step 노이즈 reset, **평가 때는 노이즈 off**(평균만) 해야 일관된 플레이. ② epsilon 제거가 여러 파일에 파급(시그니처 변경). ③ NoisyNet을 켜면 ε 스케줄 관련 하이퍼파라미터는 의미 없어진다.


## 6. Distributional RL — C51 (끝판왕)

🩹 **고치는 약점 — 기대값 하나로 뭉개기.**
$Q(s,a)$ 는 "미래 수익의 **평균**" 한 숫자다. 하지만 같은 평균이라도 "거의 항상 +5"와 "절반은 0, 절반은 +10"은 완전히 다른 상황이다. 평균만 배우면 이 **분포(불확실성) 정보**가 사라진다. 분포를 직접 배우면 학습 신호가 풍부해져 표현이 좋아진다.

⚙️ **메커니즘 — 수익의 확률분포를 예측.**
가능한 수익 값을 미리 **고정된 격자(support)** 로 정해둔다: $z_0=v_{\min}, \dots, z_{N-1}=v_{\max}$, 보통 $N=51$개라 "C51". 네트워크는 각 행동마다 이 51칸 위의 **확률분포** $p_i(s,a)$ 를 출력(softmax)한다.
$$ z_i = v_{\min} + i\,\Delta z,\quad \Delta z = \frac{v_{\max}-v_{\min}}{N-1},\qquad Q(s,a) = \sum_i z_i\, p_i(s,a) $$
(필요하면 이 기대값 $Q$로 행동을 고른다 → Double/탐험과 호환.)

🧮 **학습 — 분포 벨만 + 사영(projection) + cross-entropy.**
타깃 분포는 "보상만큼 옮기고 $\gamma$만큼 수축"시킨 것: 각 격자점이 $\hat{\mathcal{T}} z_j = r + \gamma^{n} z_j$ 로 이동. 그런데 이동한 위치는 **격자 사이**에 떨어지므로, 그 확률질량을 양 옆 격자점에 나눠주는 **사영**을 한다(이게 C51 구현에서 제일 까다로운 부분):
$$ b_j = \frac{\mathrm{clip}(\hat{\mathcal{T}} z_j,\, v_{\min}, v_{\max}) - v_{\min}}{\Delta z},\quad l=\lfloor b_j\rfloor,\ u=\lceil b_j\rceil $$
질량 $p_j(s',a^*)$ 를 아래칸 $l$에 $(u-b_j)$, 위칸 $u$에 $(b_j-l)$ 비율로 분배. 이렇게 만든 타깃 분포 $m$ 과 예측 분포의 **교차 엔트로피**가 손실:
$$ \text{loss} = -\sum_i m_i \,\log p_i(s,a) $$
(기존 smooth_l1 회귀를 **완전히 대체**.)

🔌 **우리 코드 어디가 바뀌나 (개념):** ① 출력층이 $(\text{행동수} \times N)$ logits → softmax. ② 행동 선택은 기대값 $\sum z_i p_i$ 의 argmax. ③ loss 전체가 사영+cross-entropy로 교체. ④ $v_{\min}, v_{\max}, N$ 하이퍼파라미터 추가.

⚠️ **함정:** ① **사영 코드**가 핵심 난관 — 천천히, 작은 예제로 검증. ② $v_{\min}/v_{\max}$ 는 게임 수익 범위에 맞춰야(reward clip + n-step이면 유계라 표준 $-10\sim10$ 무난). ③ n-step과 만나면 $\gamma \to \gamma^{n}$. ④ Double과 만나면 다음 행동 $a^*$ 를 **온라인망의 기대값**으로 고른다.


## 7. 6개가 어떻게 합쳐지나 (상호작용 지도)

각 요소는 독립적이지만, 합칠 때 **만나는 지점**이 있다. 특히 C51이 다른 것들의 "Q값"을 "분포/기대값"으로 바꿔놓는다.

- **Double × C51**: 다음 행동 $a^*$ 를 *온라인망의 기대값* $\sum z_i p_i^{\theta}(s',\cdot)$ 으로 고르고, 그 행동의 *분포는 타깃망*에서 가져온다.
- **n-step × C51**: 분포 이동이 $r + \gamma^{n} z_j$ (할인이 $\gamma^{n}$).
- **PER × C51**: 우선순위로 쓰던 $|\text{TD오차}|$ 대신 **샘플별 cross-entropy 손실**을 우선순위로 사용.
- **Dueling × C51**: V·A 두 갈래가 각각 *분포(로짓)*를 내고, 합친 뒤 행동·격자 축으로 softmax.
- **NoisyNet × ε-greedy**: 공존 X — Noisy를 켜면 ε-greedy는 제거. 모든 FC(특히 Dueling 갈래)를 NoisyLinear로.

> **데이터 흐름 요약 (full Rainbow)**
> ```
> 환경 → 전처리(흡수) → n-step 큐 → PER 버퍼(우선순위) → 배치 샘플(+IS가중치)
>      → [Noisy+Dueling+C51 CNN] 예측 분포
>      → 타깃 분포(Double로 행동선택, n-step γ^n, 사영) → cross-entropy(×IS가중치)
>      → 역전파(온라인망) → 우선순위 갱신 → (주기적) 타깃망 복사
> ```

**그래서 점진적으로 붙이는 이유**: 한꺼번에 넣으면 위 만남 지점에서 버그가 나도 *어느 요소 탓인지* 분리가 안 된다. 하나씩 + 곡선 비교가 공부에도 디버깅에도 최적.


## 8. Seaquest 적용 포인트

- **왜 Seaquest인가**: vanilla DQN이 약한 대표 게임. 보상 구조가 풍부(적 사격 +점수, 잠수부 구출 보너스, **산소 관리**라는 장기 의존성)해서 n-step·분포 RL의 효과가 눈에 띈다. 행동 공간도 18개로 Breakout(4개)보다 흥미롭다.
- **환경 설정 차이**: Breakout과 달리 시작 FIRE 불필요(자동 시작). 표준은 `terminal_on_life_loss=True`(목숨 잃을 때마다 종료 신호 → 학습 신호 ↑). reward clipping은 유지.
- **수익 범위와 C51**: reward clip + n-step이면 한 전이 타깃이 유계라 $v_{\min}=-10,\ v_{\max}=10,\ N=51$ 이 무난한 출발점.
- **게임 교체는 한 줄**: flag/설정 구조라 `ALE/Seaquest-v5` → `ALE/Enduro-v5` 문자열만 바꾸면 Enduro로도 동일 코드로 실험 가능.
- **현실적 비용**: 장기 학습(수 시간~하루). 그래서 **배선 검증은 Pong으로 빠르게**, 본 실험만 Seaquest로(아래 9번).


## 9. 공부 → 구현 순서 & 검증 전략

**공부 순서 (이 노트북)**: 0 → 1 → 2 → 3 → 4 → 5 → 6 → 7. 각 절의 🧮수식과 ⚠️함정만이라도 손으로 한 번 정리해보면 구현 때 훨씬 빠르다.

**구현 순서 (plan 파일과 동일, 난이도/효과 순)**:
1. (Phase 0) 토대 — flag/설정, 모델 팩토리, 체크포인트에 설정 저장, 환경 일반화
2. Double → 3. Dueling → 4. n-step → 5. PER → 6. NoisyNet → 7. C51

**단계마다 같은 리듬으로 검증**:
- ✅ **스모크 런**(~2k step): 크래시·NaN 없는지.
- ✅ **빠른 방향 확인은 Pong**: Seaquest 풀런은 비싸니, 새 요소 붙일 때마다 Pong으로 짧게 "수렴 방향"만.
- ✅ **본 실험(Seaquest)**: 구성별 체크포인트 저장 → 학습곡선.
- ✅ **건강 지표**: episode return 우상향 / loss 폭주·NaN 없음 / (Double) 예측 Q 평균 안정.
- ✅ **마무리 ablation**: vanilla → +Double → +Dueling → … → full Rainbow 곡선을 **한 그래프에 겹쳐** 기여도 정리(발표/복습용).

**흔한 함정 모음(요약)**: n-step 에피소드 경계 / PER β 어닐링·메모리 / Noisy 평가 시 노이즈 off·epsilon 제거 파급 / C51 사영 코드·$v_{\min,\max}$ 범위 / Double·C51 결합 시 기대값 argmax.


## 10. 참고자료 & 셀프 체크리스트

**논문 (막힐 때 해당 부분만 대조)**
- Rainbow: Hessel et al. 2018 (6개 통합)
- Double DQN: van Hasselt et al. 2016
- Dueling: Wang et al. 2016
- PER: Schaul et al. 2016
- n-step: Sutton & Barto (TD(λ)/n-step return 장)
- C51: Bellemare et al. 2017 ("A Distributional Perspective on RL")
- NoisyNet: Fortunato et al. 2018

**구현 대조용 (통째 복붙 X, 막힌 부분만)**
- CleanRL — 단일 파일이라 읽기 쉬움
- Google Dopamine — `rainbow_agent`
- Stable-Baselines3

**셀프 체크 — 각 요소를 "왜·어떻게"로 설명할 수 있나?**
- [ ] Double이 과대추정을 왜 줄이나? (선택/평가 분리)
- [ ] Dueling에서 평균을 왜 빼나? (식별성)
- [ ] n-step에서 $\gamma^{n}$ 이 왜 나오나? + 에피소드 경계 처리
- [ ] PER의 IS 가중치는 무엇을 보정하나? + sum-tree의 역할
- [ ] NoisyNet은 무엇을 학습해서 탐험을 조절하나? ($\sigma$)
- [ ] C51의 사영(projection)은 왜 필요하나? (이동한 질량이 격자 사이에 떨어져서)
- [ ] 6개가 만나는 지점 5개를 말할 수 있나? (7번)

> 다음 단계: 이 노트북으로 그림이 잡히면, plan 파일의 **Phase 0 → 1**부터 직접 구현 시작.
